# HealthGait — Quick-Start Guide

This notebook covers the full usage cycle:
1. Download the pretrained model from HuggingFace
2. Inspect preprocessing and model configuration
3. Reconstruct and load the model (architecture auto-detected from the checkpoint)
4. Extract embeddings from skeleton sequences
5. Simulate **continuing training** (optimizer + scheduler + loss loop)
6. Run an **evaluation pass** (reconstruction loss)

**Model**: [`adamthe1/HealthGait`](https://huggingface.co/adamthe1/HealthGait) — epoch 31  
**Input**: skeleton sequences of shape `[batch, frames, 26 joints, 4 channels (x, y, z, confidence)]`

> Sections 4–6 use **synthetic data** to demonstrate the API. Swap in your real DataLoader batches where indicated.

## 0  Setup

In [ ]:
# Uncomment if any package is missing
# !pip install torch huggingface_hub

In [ ]:
import ast, json, sys, os

# Resolve project root — works whether the Jupyter CWD is model/ or the repo root
_cwd = os.getcwd()
if os.path.basename(_cwd) == "model":
    project_root = os.path.dirname(_cwd)
elif os.path.isdir(os.path.join(_cwd, "model", "architecture")):
    project_root = _cwd
else:
    project_root = os.path.abspath(os.path.join(_cwd, ".."))

if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f"Project root: {project_root}")

import torch
import numpy as np
from huggingface_hub import hf_hub_download

from model.architecture.motionBert_full import DSTformer, ReconstructNet
from model.preprocessing.args import PreprocessingArgs
from model.training.utils.training_helper import get_scheduler, load_checkpoint
from model.training.utils.loss import loss_mpjpe, loss_velocity

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## 1  Download from HuggingFace

Files are cached in the default HuggingFace cache (`~/.cache/huggingface/hub/`).  
Re-running this cell is a no-op if the files are already cached.

In [ ]:
REPO_ID = "adamthe1/HealthGait"

checkpoint_path  = hf_hub_download(repo_id=REPO_ID, filename="epoch_31.pth")
prep_args_path   = hf_hub_download(repo_id=REPO_ID, filename="preprocessing_args.json")
model_args_path  = hf_hub_download(repo_id=REPO_ID, filename="model_args.txt")

print(f"Checkpoint       : {checkpoint_path}")
print(f"Preprocessing cfg: {prep_args_path}")
print(f"Model args       : {model_args_path}")

## 2  Inspect Configuration

In [ ]:
# ── Preprocessing args ────────────────────────────────────────────────────────
with open(prep_args_path) as f:
    prep_data = json.load(f)
prep_args = PreprocessingArgs(**prep_data)

print("Preprocessing configuration")
print(f"  Activities : {list(prep_args.videos_lens.keys())}")
print(f"  Excluded   : {prep_args.exclude_list}")
print(f"  Joints     : {prep_args.get_num_joints()}  (remove_noise_joints={prep_args.remove_noise_joints})")
print(f"  Augments on: {[k for k, v in prep_args.augments.items() if v]}")
print(f"  Group mask : {prep_args.group_masking}  span={prep_args.span_masking}")
print(f"  Centroid   : {prep_args.centroid_type}")

In [ ]:
# ── Model (training) args ─────────────────────────────────────────────────────
with open(model_args_path) as f:
    model_args = ast.literal_eval(f.read())

print("Model / training configuration")
for k, v in model_args.items():
    print(f"  {k}: {v}")

## 3  Build and Load Model

Architecture parameters (depth, `dim_feat`, RoPE, sink tokens …) are **auto-detected**
from the checkpoint's weight shapes — no manual config required.

In [ ]:
# ── Load raw checkpoint ───────────────────────────────────────────────────────
raw_ckpt = torch.load(checkpoint_path, map_location="cpu")
state    = raw_ckpt["model"]

# Strip torch.compile and DataParallel prefixes if present
if any(k.startswith("_orig_mod.") for k in state):
    state = {k[len("_orig_mod."):]: v for k, v in state.items()}
if any(k.startswith("module.") for k in state):
    state = {k[len("module."):]: v for k, v in state.items()}

print(f"Checkpoint epoch: {raw_ckpt['epoch']}")
print(f"State dict keys (sample): {list(state.keys())[:6]}")

In [ ]:
# ── Detect architecture from weight shapes ────────────────────────────────────
def detect_arch(sd):
    dim_feat       = sd["model_backbone.pos_embed"].shape[-1]
    num_joints     = sd["model_backbone.pos_embed"].shape[1]
    depth          = sum(1 for k in sd
                         if "model_backbone.blocks_st." in k
                         and k.endswith(".norm1_s.weight"))
    if "model_backbone.pre_logits.fc.weight" in sd:
        dim_rep    = sd["model_backbone.pre_logits.fc.weight"].shape[0]
    else:
        dim_rep    = dim_feat   # Identity pre_logits
    # RoPE: when enabled, temp_embed is a None buffer and is absent from state_dict
    with_rope      = "model_backbone.temp_embed" not in sd
    num_sink       = (sd["model_backbone.sink_tokens"].shape[1]
                      if "model_backbone.sink_tokens" in sd else 0)
    # Linear weight shape is [out_features, in_features]
    dim_in         = sd["joint_embd.weight"].shape[1]
    dim_out        = sd["reconstruct_head.weight"].shape[0]
    return dict(dim_feat=dim_feat, num_joints=num_joints, depth=depth,
                dim_rep=dim_rep, with_rope=with_rope, num_sink_tokens=num_sink,
                dim_in=dim_in, dim_out=dim_out)

arch = detect_arch(state)
print("Detected architecture:")
for k, v in arch.items():
    print(f"  {k}: {v}")

In [ ]:
# ── Instantiate backbone + wrapper ────────────────────────────────────────────
backbone = DSTformer(
    num_joints      = arch["num_joints"],
    dim_in          = arch["dim_in"],
    maxlen          = model_args["size_seq"],
    depth           = arch["depth"],
    drop_rate       = model_args["dropout_ratio"],
    use_rope        = arch["with_rope"],
    use_flash_attn  = False,   # disable for portability; weights are unaffected
    dim_feat        = arch["dim_feat"],
    dim_rep         = arch["dim_rep"],
    num_heads       = model_args["num_heads"],
    num_sink_tokens = arch["num_sink_tokens"],
)

model = ReconstructNet(
    backbone,
    dim_in  = arch["dim_in"],
    dim_out = arch["dim_out"],
)

model.load_state_dict(state)
model = model.to(device).eval()

n_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded — {n_params / 1e6:.2f}M parameters")

## 4  Embedding Extraction (Inference)

Input skeleton sequences have shape `[B, T, J, C]`:
- `B` — batch size
- `T` — frames (model was trained with `T=900`; shorter windows work too)
- `J` — 26 joints
- `C` — 4 channels: `(x, y, z, confidence)`

The model returns:
- `reconstructed` `[B, T, J, 3]` — predicted 3-D joint positions
- `bundle['pooled_pre_logits_embeddings']` `[B, dim_rep × 3]` — subject-level embedding (mean + max + std pooled)

In [ ]:
# Synthetic batch — replace with your real skeleton tensors
DEMO_T = 64   # short window for fast demo; use model_args['size_seq'] (900) for full-length
B = 4

x_demo = torch.randn(B, DEMO_T, arch["num_joints"], arch["dim_in"], device=device)

with torch.no_grad():
    reconstructed, bundle = model(x_demo)

embeddings = bundle["pooled_pre_logits_embeddings"]   # [B, dim_rep * 3]

print(f"Input             : {tuple(x_demo.shape)}")
print(f"Reconstructed XYZ : {tuple(reconstructed.shape)}")
print(f"Subject embedding : {tuple(embeddings.shape)}  ← use this for downstream tasks")

## 5  Continue Training

This mirrors the flow in `model/training/continue_training.py` and `train_loop.py`.

- **Optimizer**: AdamW with the same LR and weight-decay as the original run
- **Scheduler**: cosine-annealing with a short warmup (via `get_scheduler`)
- **Loss**: MPJPE reconstruction + optional velocity term

Uncomment the `load_checkpoint` call to also restore the optimizer and scheduler states
for a true resumption (no learning-rate reset).

In [ ]:
# ── Optimizer & scheduler ─────────────────────────────────────────────────────
model.train()

optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr           = model_args["learning_rate"],
    weight_decay = model_args["adamw_weight_decay"],
    fused        = (device.type == "cuda"),
)

STEPS_PER_EPOCH = 200   # set to len(train_loader) when using real data

scheduler = get_scheduler(optimizer, {
    "scheduler_type"  : model_args["scheduler_type"],
    "num_epochs"      : model_args["num_epochs"],
    "warmup_epochs"   : model_args["warmup_epochs"],
    "warmup_ratio"    : model_args["warmup_ratio"],
    "steps_per_epoch" : STEPS_PER_EPOCH,
    "learning_rate"   : model_args["learning_rate"],
})

# Optional — restore optimizer + scheduler state for exact resumption:
# model, optimizer, scheduler, scaler, start_epoch = load_checkpoint(
#     model, optimizer, scheduler, checkpoint_path, device
# )

start_epoch = raw_ckpt["epoch"] + 1
print(f"Ready. Resuming from epoch {start_epoch} / {model_args['num_epochs']}")
print(f"LR = {optimizer.param_groups[0]['lr']:.2e}")

In [ ]:
# ── Simulated training steps ──────────────────────────────────────────────────
# Replace the synthetic tensors below with batches from your DataLoader:
#   for batch in train_loader:
#       x      = batch['data'].to(device)             # [B, T, J, C]
#       target = batch['original'][..., :3].to(device) # [B, T, J, 3]  (XYZ)

lambda_vel = model_args["lambda_3d_velocity"]   # 0.1

DEMO_BATCH = 2

for step in range(1, 6):
    # ── synthetic data (swap with real loader) ────────────────────────────────
    x      = torch.randn(DEMO_BATCH, DEMO_T, arch["num_joints"], arch["dim_in"],  device=device)
    target = torch.randn(DEMO_BATCH, DEMO_T, arch["num_joints"], 3,               device=device)
    # ─────────────────────────────────────────────────────────────────────────

    optimizer.zero_grad()
    reconstructed, _ = model(x)
    pred_xyz = reconstructed[..., :3]

    loss = loss_mpjpe(pred_xyz, target)
    if lambda_vel > 0:
        loss = loss + lambda_vel * loss_velocity(pred_xyz, target)

    loss.backward()
    optimizer.step()
    scheduler.step()

    print(f"  step {step}  loss={loss.item():.4f}  lr={scheduler.get_last_lr()[0]:.2e}")

## 6  Evaluation Pass

Mirrors `_collect_model_outputs` in `training_helper.py`.  
Reports mean per-joint position error (MPJPE) in the same units as the input coordinates.

In [ ]:
model.eval()

eval_losses  = []
all_embeddings = []

# Replace range(...) with your eval_loader:
#   for batch in eval_loader:
#       x      = batch['data'].to(device)
#       target = batch['original'][..., :3].to(device)

DEMO_EVAL_STEPS = 8

with torch.no_grad():
    for _ in range(DEMO_EVAL_STEPS):
        # ── synthetic data ────────────────────────────────────────────────────
        x      = torch.randn(DEMO_BATCH, DEMO_T, arch["num_joints"], arch["dim_in"],  device=device)
        target = torch.randn(DEMO_BATCH, DEMO_T, arch["num_joints"], 3,               device=device)
        # ─────────────────────────────────────────────────────────────────────

        reconstructed, bundle = model(x)
        pred_xyz = reconstructed[..., :3]

        eval_losses.append(loss_mpjpe(pred_xyz, target).item())
        all_embeddings.append(bundle["pooled_pre_logits_embeddings"].cpu().numpy())

embeddings_array = np.concatenate(all_embeddings, axis=0)   # [N_samples, embed_dim]

print(f"Eval MPJPE (reconstruction) : {np.mean(eval_losses):.4f}")
print(f"Collected embeddings shape  : {embeddings_array.shape}")
print("\nEmbeddings are ready for downstream probing (e.g. ridge regression, logistic regression).")